# Chimera v2.4 — Data Preparation

Single source of truth for training data creation. Run top-to-bottom from a clean clone.

**Pipeline overview:**

1. Download the CAMEX pancreas dataset + van Zyl aqueous humor (raw UMI counts).
2. Query Ensembl BioMart for 1:1 ortholog tables across all species.
3. Compute gene-level features: conservation prior, ESM protein embeddings,
   PCA reduction, inter-species variance, and expression statistics.
4. Construct a unified cross-species AnnData aligned by ortholog groups.
5. Precompute null expression distributions and run pre-training quality checks.
6. Save the final `.h5ad` to disk with all required fields.

| Metric | Estimate |
|---|---|
| **Runtime** | ~30 min (network-dependent; downloads cached for re-runs) |
| **Disk** | ~10 GB (raw downloads + processed outputs) |
| **Prerequisites** | `pip install -e ".[data]"` (adds `gdown` for CAMEX download) |

In [ ]:
import logging
import pickle
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from scipy import stats

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

from chimera._data_utils import (
    assemble_gin_features,
    compute_conservation_prior,
    compute_expression_stats,
    compute_esm2_interspecies_variance,
    construct_unified_adata,
    download_and_process_esm2,
    download_camex_pancreas,
    download_camex_testis,
    download_van_zyl,
    pca_esm2,
    precompute_p_null,
    query_ortholog_table,
)
from chimera._diagnostics import (
    library_size_heterogeneity_diagnostic,
    cell_type_composition_diagnostic,
)

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Resolve paths relative to this notebook
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"Raw data dir  : {RAW_DIR.resolve()}")
print(f"Processed dir : {PROCESSED_DIR.resolve()}")

---
## 1. Download Raw Data

Load the CAMEX pancreas atlas, which contains raw UMI counts for 4--6
mammalian species (human, mouse, rat, pig, macaque, marmoset) with
cell-type and batch annotations.

On the first run, the data is downloaded from the CAMEX consortium portal
and cached locally.  Subsequent runs load from the cache.

In [ ]:
adata_pancreas = download_camex_pancreas(data_dir=RAW_DIR)

print(f"CAMEX pancreas: {adata_pancreas.n_obs} cells, {adata_pancreas.n_vars} genes")
print(f"Species: {adata_pancreas.obs['species'].value_counts().to_dict()}")
print(f"Batches: {adata_pancreas.obs['batch'].nunique()}")
print(f"Cell types: {adata_pancreas.obs['cell_type'].nunique()}")

In [ ]:
# Download van Zyl aqueous humor 5-species dataset (Task 1.2)
# Source: SATURN server (http://snap.stanford.edu/saturn/data/ah_atlas_export.tar.gz)
# Species: human, cynomolgus macaque, rhesus macaque, pig, mouse

adata_vanzyl = download_van_zyl(data_dir=RAW_DIR)

print(f"\nvan Zyl aqueous humor: {adata_vanzyl.n_obs} cells, {adata_vanzyl.n_vars} genes")
print(f"Species ({adata_vanzyl.obs['species'].nunique()}):")
for sp, cnt in adata_vanzyl.obs["species"].value_counts().items():
    n_batches = adata_vanzyl.obs.loc[adata_vanzyl.obs["species"] == sp, "batch"].nunique()
    print(f"  {sp:>25s}: {cnt:>6d} cells, {n_batches} batches")

# Validate: each batch maps to exactly one species (Task 7.2 requirement)
for b in adata_vanzyl.obs["batch"].cat.categories:
    n_sp = adata_vanzyl.obs.loc[adata_vanzyl.obs["batch"] == b, "species"].nunique()
    assert n_sp == 1, f"Batch '{b}' spans {n_sp} species!"
print("\nBatch-species constraint: PASS (each batch → one species)")

---
## 2. Ortholog Mapping and Conservation Prior

Retrieve 1:1 ortholog groups across all species present in the dataset via
Ensembl BioMart. The ortholog table maps each species' gene IDs to a common
ortholog group ID and records percent identity for each non-reference species
relative to human.

In [ ]:
species_list = sorted(adata_pancreas.obs["species"].unique().tolist())
print(f"Species in dataset: {species_list}")

# Comprehensive mapping from common names to Ensembl identifiers
# Covers all 11 CAMEX testis species + van Zyl species
ENSEMBL_SPECIES = {
    "human": "hsapiens", "homo sapiens": "hsapiens",
    "mouse": "mmusculus", "mus musculus": "mmusculus",
    "macaque": "mmulatta", "rhesus macaque": "mmulatta",
    "cynomolgus macaque": "mfascicularis",
    "pig": "sscrofa", "sus scrofa": "sscrofa",
    "rat": "rnorvegicus",
    "chimpanzee": "ptroglodytes", "chimp": "ptroglodytes",
    "gorilla": "ggorilla",
    "bonobo": "ppaniscus",
    "gibbon": "nleucogenys",
    "marmoset": "cjacchus",
    "opossum": "mdomestica",
    "platypus": "oanatinus",
    "chicken": "ggallus",
}

ensembl_names = {}  # common → ensembl (for later use in per-species split)
ensembl_species = []
for sp in species_list:
    sp_lower = sp.lower().strip()
    if sp_lower in ENSEMBL_SPECIES:
        ens = ENSEMBL_SPECIES[sp_lower]
    else:
        logging.warning("No Ensembl mapping for '%s' — using as-is", sp)
        ens = sp_lower
    ensembl_names[sp] = ens
    ensembl_species.append(ens)

print(f"Ensembl species: {ensembl_species}")

ortholog_table = query_ortholog_table(
    species_list=ensembl_species,
    reference_species="hsapiens",
    cache_dir=PROCESSED_DIR,
)
print(f"\n1:1 ortholog groups: {len(ortholog_table)}")
print(f"Mean identity: {ortholog_table['mean_identity'].mean():.1f}%")
print(f"Median identity: {ortholog_table['mean_identity'].median():.1f}%")

# Log intersection size for Caveat 25 (ascertainment bias)
print(f"\nNote (Caveat 25): 1:1 ortholog restriction enriches for slowly evolving,")
print(f"functionally constrained genes. Intersection size = {len(ortholog_table)} genes.")

In [ ]:
# --- Ortholog QC: pairwise ortholog counts and identity distributions ---

non_ref_species = [s for s in ensembl_species if s != "hsapiens"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Number of 1:1 orthologs contributed by each pairwise comparison
pair_counts = {}
for sp in non_ref_species:
    id_col = f"{sp}_gene_id"
    if id_col in ortholog_table.columns:
        pair_counts[sp] = ortholog_table[id_col].notna().sum()

axes[0].barh(list(pair_counts.keys()), list(pair_counts.values()), color="steelblue")
axes[0].set_xlabel("Number of 1:1 orthologs")
axes[0].set_title("Pairwise 1:1 ortholog count (vs. human)")
for i, (sp, cnt) in enumerate(pair_counts.items()):
    axes[0].text(cnt + 50, i, str(cnt), va="center", fontsize=9)

# (b) Distribution of mean percent identity across ortholog groups
axes[1].hist(
    ortholog_table["mean_identity"].dropna(),
    bins=50,
    color="coral",
    edgecolor="white",
)
axes[1].axvline(
    ortholog_table["mean_identity"].median(),
    color="black",
    linestyle="--",
    label=f"median = {ortholog_table['mean_identity'].median():.1f}%",
)
axes[1].set_xlabel("Mean percent identity")
axes[1].set_ylabel("Number of ortholog groups")
axes[1].set_title("Ortholog identity distribution")
axes[1].legend()

fig.tight_layout()
plt.show()

---
## 3. Gene Features

Chimera v2.4 uses a Gene Interaction Network (GIN) that consumes per-gene
feature vectors to learn the shared/residual mixing coefficient (alpha).
The feature vector has ~72 dimensions:

| Feature block | Dims | Source |
|---|---|---|
| ESM-2 PCA | 64 | SATURN ESM-2 3B protein embeddings, PCA-reduced |
| ESM-2 inter-species variance | 1 | Mean pairwise cosine distance across species |
| Mean identity | 1 | From ortholog table |
| Expression stats | 3 | log mean, log variance, fraction non-zero |
| Padding | 3 | Zeros (reserved for future features) |

In [ ]:
# 3a. Conservation prior from ortholog percent identity
conservation_prior = compute_conservation_prior(ortholog_table)

print(f"Conservation prior shape: {conservation_prior.shape}")
print(f"  range: [{conservation_prior.min().item():.3f}, {conservation_prior.max().item():.3f}]")
print(f"  mean:  {conservation_prior.mean().item():.3f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(conservation_prior.numpy(), bins=50, color="mediumpurple", edgecolor="white")
ax.set_xlabel("Conservation prior (clamped identity / 100)")
ax.set_ylabel("Count")
ax.set_title("Conservation prior distribution")
fig.tight_layout()
plt.show()

In [ ]:
# 3b. ESM-2 protein embeddings: download, consensus, PCA, inter-species variance

# Download per-species ESM-2 embeddings from SATURN and compute consensus
consensus_emb = download_and_process_esm2(
    ortholog_table,
    species_list=ensembl_species,
    saturn_data_dir=RAW_DIR / "saturn_esm2",
)
print(f"Consensus ESM-2 embeddings: {consensus_emb.shape}")

# PCA: 5120 -> 64 dimensions
esm2_pca = pca_esm2(consensus_emb, n_components=64)
print(f"ESM-2 PCA: {esm2_pca.shape}")

# Load per-species embeddings for inter-species variance computation
saturn_dir = RAW_DIR / "saturn_esm2"
per_species_emb = {}
for sp in ensembl_species:
    emb_path = saturn_dir / f"{sp}_esm2.npy"
    if emb_path.exists():
        sp_data = np.load(emb_path, allow_pickle=True).item()
        gene_id_col = f"{sp}_gene_id"
        if gene_id_col in ortholog_table.columns:
            gene_ids = ortholog_table[gene_id_col].values
            emb_matrix = np.zeros((len(ortholog_table), 5120), dtype=np.float32)
            for i, gid in enumerate(gene_ids):
                if gid in sp_data:
                    emb_matrix[i] = sp_data[gid]
            per_species_emb[sp] = emb_matrix

esm2_interspecies_var = compute_esm2_interspecies_variance(per_species_emb)
print(f"ESM-2 inter-species variance: {esm2_interspecies_var.shape}")
print(f"  mean cosine dist: {esm2_interspecies_var.mean():.4f}")

In [ ]:
# 3c. Expression stats and GIN feature assembly
#
# Expression statistics require the unified AnnData (computed in Section 4).
# We store the partial features here and complete assembly after unification.

print("Expression stats and full GIN feature assembly deferred to Section 4.")
print(f"Partial features ready:")
print(f"  esm2_pca             : {esm2_pca.shape}")
print(f"  esm2_interspecies_var: {esm2_interspecies_var.shape}")
print(f"  conservation_prior   : {conservation_prior.shape}")

---
## 4. Unified AnnData Construction

Each species' expression matrix uses its own gene namespace.  Here we:

1. Split the raw data into per-species AnnData objects.
2. Map each species' genes to ortholog group IDs using the ortholog table.
3. Concatenate into a single unified AnnData where columns = ortholog groups.
4. Compute expression statistics on the unified data.
5. Assemble and attach the full GIN feature matrix.

In [ ]:
# 4a. Split raw data into per-species AnnDatas
per_species_adatas = {}
for sp_common in species_list:
    sp_mask = adata_pancreas.obs["species"] == sp_common
    sp_adata = adata_pancreas[sp_mask].copy()
    # Use the Ensembl species name as the key for ortholog table matching
    ensembl_name = ensembl_names[sp_common]
    per_species_adatas[ensembl_name] = sp_adata
    print(f"  {sp_common:>10s} ({ensembl_name}): {sp_adata.n_obs:>6d} cells, {sp_adata.n_vars:>6d} genes")

print(f"\nTotal species: {len(per_species_adatas)}")

In [ ]:
# 4b. Construct unified AnnData aligned by ortholog groups
#
# We pass placeholder gene features here; the real features are attached
# after computing expression statistics on the unified data.

G = len(ortholog_table)
placeholder_features = np.zeros((G, 72), dtype=np.float32)

unified_adata = construct_unified_adata(
    per_species_adatas=per_species_adatas,
    ortholog_table=ortholog_table,
    gene_features=placeholder_features,
    conservation_prior=conservation_prior,
)

print(f"Unified AnnData: {unified_adata.n_obs} cells x {unified_adata.n_vars} genes (ortholog groups)")
print(f"Species: {unified_adata.obs['species'].value_counts().to_dict()}")
print(f"Batches: {unified_adata.obs['batch'].nunique()}")

In [ ]:
# 4c. Compute expression statistics on unified data and assemble GIN features

expression_stats = compute_expression_stats(unified_adata)
print("Expression statistics:")
for key, arr in expression_stats.items():
    print(f"  {key:>15s}: shape={arr.shape}, mean={arr.mean():.4f}, std={arr.std():.4f}")

gene_features, gin_scaler = assemble_gin_features(
    esm2_pca=esm2_pca,
    esm2_interspecies_var=esm2_interspecies_var,
    ortholog_table=ortholog_table,
    expression_stats=expression_stats,
)
print(f"\nGIN features: {gene_features.shape} (StandardScaled)")
print(f"  per-dim mean ~ {gene_features.mean(axis=0).mean():.6f} (should be ~0)")
print(f"  per-dim std  ~ {gene_features.std(axis=0).mean():.6f} (should be ~1)")

# Replace placeholder features in the unified AnnData
unified_adata.varm["gene_features"] = gene_features

---
## 5. p_null and Quality Checks

Before training, we:

1. Precompute the **null expression distribution** (Laplace-smoothed gene
   proportions per species) used by the NB reconstruction loss.
2. Run **library-size heterogeneity** diagnostics to decide whether to use
   observed or learned library sizes.
3. Run **cell-type composition** diagnostics to flag species with dominant
   cell types that could bias integration.
4. Run the **pre-training conservation signal** check to confirm the
   conservation prior correlates with expression pattern similarity.

In [ ]:
# 5a. Precompute p_null
p_null = precompute_p_null(unified_adata)

n_species_null = p_null.shape[0]
print(f"p_null shape: {p_null.shape}  (K={n_species_null} species x G={p_null.shape[1]} genes)")
print(f"Row sums (should be ~1): {p_null.sum(axis=1)}")

fig, ax = plt.subplots(figsize=(8, 3))
for s_idx in range(n_species_null):
    ax.hist(
        np.log10(p_null[s_idx] + 1e-12),
        bins=80,
        alpha=0.5,
        label=f"species {s_idx}",
    )
ax.set_xlabel("log10(p_null)")
ax.set_ylabel("Count")
ax.set_title("Null expression distribution per species")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# 5b. Library-size heterogeneity diagnostic
from chimera._diagnostics import library_size_heterogeneity_diagnostic

lib_diag = library_size_heterogeneity_diagnostic(unified_adata)

print(f"Library size ratio: {lib_diag['max_min_ratio']:.2f}x")
print(f"Recommendation: {lib_diag['recommendation']}")
print(f"Use observed lib size: {lib_diag['use_observed_lib_size']}")
print(f"Details: {lib_diag['details']}")
print(f"\nPer-species median library sizes:")
for sp_name, med_lib in sorted(lib_diag["per_species_median_lib"].items()):
    print(f"  {sp_name:>10s}: {med_lib:,.0f}")

In [ ]:
# 5c. Cell-type composition diagnostic

comp_diag = cell_type_composition_diagnostic(unified_adata)

print(f"Flagged species: {comp_diag['flagged_species']}")
print(f"Details: {comp_diag['details']}")

In [ ]:
# 5d. Pre-training conservation signal check (Task 9.9 preview)
#
# Before investing in full model training, verify that conservation features
# have predictive power for expression similarity in this dataset.

species_vals = unified_adata.obs["species"].values
unique_sp = np.unique(species_vals)
X = unified_adata.X
if hasattr(X, "toarray"):
    X_dense = X.toarray()
else:
    X_dense = np.asarray(X)

# Per-gene cross-species expression similarity (1 - CV of species means)
sp_means = np.stack([X_dense[species_vals == sp].mean(axis=0) for sp in unique_sp])  # (K, G)
gene_sim = np.array([
    1.0 - sp_means[:, g].std() / (sp_means[:, g].mean() + 1e-8)
    for g in range(unified_adata.n_vars)
])

# Conservation prior
prior_vals = unified_adata.var["conservation_prior"].values

r_global, p_global = stats.spearmanr(prior_vals, gene_sim)
print(f"Conservation signal check:")
print(f"  Spearman r = {r_global:.4f}  (p = {p_global:.2e})")
print(f"  Decision: {'PROCEED — conservation predicts expression similarity' if r_global > 0.2 else 'INVESTIGATE — weak conservation signal'}")
print(f"\n  Interpretation:")
print(f"    r > 0.2 : conservation is informative, proceed with full feature vector")
print(f"    r ~ 0   : conservation is uninformative, consider ESM-2-only features")

---
## 6. Save and Verify

Write the unified AnnData to disk and verify that all required fields are
present. Run a smoke test with `ChimeraModel.setup_anndata()` to confirm
the output is training-ready.

In [ ]:
# Save p_null alongside unified AnnData
unified_adata.uns["p_null"] = p_null

# Save GIN scaler for inference
with open(PROCESSED_DIR / "gin_scaler.pkl", "wb") as f:
    pickle.dump(gin_scaler, f)

output_path = PROCESSED_DIR / "camex_pancreas_unified.h5ad"
unified_adata.write_h5ad(output_path)
print(f"Saved unified AnnData to {output_path}")
print(f"File size: {output_path.stat().st_size / 1e6:.1f} MB")

# --- Assertions: verify all required fields ---
assert "species" in unified_adata.obs.columns, "Missing obs['species']"
assert "batch" in unified_adata.obs.columns, "Missing obs['batch']"
assert "gene_features" in unified_adata.varm, "Missing varm['gene_features']"
assert "conservation_prior" in unified_adata.var.columns, "Missing var['conservation_prior']"
assert "p_null" in unified_adata.uns, "Missing uns['p_null']"

print("\nField verification:")
print(f"  obs['species']             : {unified_adata.obs['species'].nunique()} unique")
print(f"  obs['batch']               : {unified_adata.obs['batch'].nunique()} unique")
print(f"  varm['gene_features']      : {unified_adata.varm['gene_features'].shape}")
print(f"  var['conservation_prior']  : {unified_adata.var['conservation_prior'].shape}")
print(f"  uns['p_null']              : {unified_adata.uns['p_null'].shape}")
print(f"  .X shape                   : {unified_adata.X.shape}")

# --- Smoke test: ChimeraModel.setup_anndata + construction ---
from chimera import ChimeraModel

adata_test = ad.read_h5ad(output_path)
ChimeraModel.setup_anndata(adata_test, species_key="species", batch_key="batch")
model = ChimeraModel(adata_test)
alpha = model.get_alpha()
print(f"\nChimeraModel smoke test:")
print(f"  Alpha shape: {alpha.shape}")
print(f"  Alpha range: [{alpha.min():.4f}, {alpha.max():.4f}]")
print(f"  Alpha mean:  {alpha.mean():.4f}")

print("\nAll assertions passed. Data preparation complete.")

---
## Summary

| Output | Path | Description |
|--------|------|-------------|
| Unified AnnData | `data/processed/camex_pancreas_unified.h5ad` | Training-ready multi-species data |
| Ortholog table | `data/processed/ortholog_table.csv` | 1:1 ortholog groups with identity stats |
| GIN features | Stored in `adata.varm["gene_features"]` | StandardScaled (G, 72) feature matrix |
| Conservation prior | Stored in `adata.var["conservation_prior"]` | Per-gene prior in [0.05, 0.95] |
| p_null | Stored in `adata.uns["p_null"]` | Null expression rates (K, G) |

**Next step**: Open `02_reproducibility.ipynb` to train the Chimera v2.4 model on this data.